In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum as spark_sum, count, window
import os
import shutil
import time
import pandas as pd
from datetime import datetime, timedelta

# Inisialisasi SparkSession sesuai instruksi
spark = SparkSession.builder \
    .appName("Tugas10_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4") # Mempercepat pemrosesan lokal

# Menyiapkan folder simulasi dan output
folder_stream = "tugas10_stream"
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

for folder in [folder_stream, folder_output, folder_checkpoint]:
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder, exist_ok=True)
    
print("Persiapan selesai. Spark siap digunakan.")

Persiapan selesai. Spark siap digunakan.


In [10]:
# Mendefinisikan skema secara eksplisit sesuai instruksi
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Membaca aliran data dari folder tugas10_stream
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load("tugas10_stream")

print("Streaming DataFrame Bagian A berhasil didefinisikan (baru rencana, belum berjalan).")
print("df_stream.isStreaming:", df_stream.isStreaming)

Streaming DataFrame Bagian A berhasil didefinisikan (baru rencana, belum berjalan).
df_stream.isStreaming: True


In [11]:
# Menambahkan kolom total_harga (qty x harga_satuan)
df_cabang = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi berdasarkan nama_cabang
agregasi_cabang = df_cabang.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Memulai streaming query untuk agregasi cabang
query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query Bagian B (Agregasi Cabang) dimulai!")
print("Status aktif:", query_cabang.isActive)

Streaming query Bagian B (Agregasi Cabang) dimulai!
Status aktif: True


In [8]:
# Menghentikan semua kueri yang masih aktif
for q in spark.streams.active:
    q.stop()
    print(f"Kueri '{q.name}' berhasil dihentikan.")

Kueri 'agregasi_cabang' berhasil dihentikan.


In [12]:
# Agregasi berdasarkan window waktu 2 menit dari kolom waktu_pesan
agregasi_waktu = df_cabang.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("pendapatan_per_window")
)

# Memulai streaming query kedua dari sumber data yang sama
query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query Bagian C (Agregasi Window Waktu) dimulai!")
print("Status aktif:", query_waktu.isActive)

Streaming query Bagian C (Agregasi Window Waktu) dimulai!
Status aktif: True


In [18]:
waktu_sekarang = datetime.now()

# --- BATCH 1 ---
pd.DataFrame({
    "pesanan_id": ["P1", "P2"],
    "nama_cabang": ["Cabang Jakarta", "Cabang Bandung"],
    "nama_menu": ["Kopi Susu", "Espresso"],
    "qty": [2, 1],
    "harga_satuan": [20000, 15000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv("tugas10_stream/batch_1.csv", index=False)

print("Batch 1:")
time.sleep(8)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# --- BATCH 2 ---
pd.DataFrame({
    "pesanan_id": ["P3", "P4"],
    "nama_cabang": ["Cabang Jakarta", "Cabang Surabaya"],
    "nama_menu": ["Latte", "Kopi Hitam"],
    "qty": [1, 3],
    "harga_satuan": [25000, 10000],
    "waktu_pesan": [(waktu_sekarang + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv("tugas10_stream/batch_2.csv", index=False)

print("Batch 2 (Akumulasi dari Batch 1 & 2):")
time.sleep(8)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# --- BATCH 3 ---
pd.DataFrame({
    "pesanan_id": ["P5"],
    "nama_cabang": ["Cabang Bandung"],
    "nama_menu": ["Americano"],
    "qty": [2],
    "harga_satuan": [18000],
    "waktu_pesan": [(waktu_sekarang + timedelta(minutes=2)).strftime("%Y-%m-%d %H:%M:%S")] * 1
}).to_csv("tugas10_stream/batch_3.csv", index=False)

print("Batch 3 (Akumulasi keseluruhan):")
time.sleep(8)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# --- Menampilkan Tabel Agregasi Window ---
print("Hasil Window 2 Menit:")
spark.sql("SELECT window.start, window.end, pendapatan_per_window FROM agregasi_waktu ORDER BY window.start").show(truncate=False)

Batch 1:
+---------------+----------------+--------------+
|    nama_cabang|total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
| Cabang Jakarta|           65000|             2|
| Cabang Bandung|           51000|             2|
|Cabang Surabaya|           30000|             1|
+---------------+----------------+--------------+

Batch 2 (Akumulasi dari Batch 1 & 2):
+---------------+----------------+--------------+
|    nama_cabang|total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
| Cabang Jakarta|           65000|             2|
| Cabang Bandung|           51000|             2|
|Cabang Surabaya|           30000|             1|
+---------------+----------------+--------------+

Batch 3 (Akumulasi keseluruhan):
+---------------+----------------+--------------+
|    nama_cabang|total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
| Cabang Jakarta|           65000|             2|
| Cabang Bandung| 

In [19]:
query_parquet = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

# Mengirim BATCH 4
pd.DataFrame({
    "pesanan_id": ["P6"],
    "nama_cabang": ["Cabang Bali"],
    "nama_menu": ["Cold Brew"],
    "qty": [1],
    "harga_satuan": [30000],
    "waktu_pesan": [(waktu_sekarang + timedelta(minutes=3)).strftime("%Y-%m-%d %H:%M:%S")] * 1
}).to_csv("tugas10_stream/batch_4.csv", index=False)

print("Batch 4 dikirim ke Parquet. Menunggu proses selesai...")
time.sleep(10)

# Verifikasi file tersimpan dengan membaca data batch biasa
df_baca_parquet = spark.read.parquet("tugas10_output")
df_baca_parquet.show()

Batch 4 dikirim ke Parquet. Menunggu proses selesai...
+----------+---------------+----------+---+------------+-------------------+
|pesanan_id|    nama_cabang| nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+---------------+----------+---+------------+-------------------+
|        P5| Cabang Bandung| Americano|  2|       18000|2026-11-05 16:57:39|
|        P6|    Cabang Bali| Cold Brew|  1|       30000|2026-11-05 16:58:39|
|        P1| Cabang Jakarta| Kopi Susu|  2|       20000|2026-11-05 16:55:39|
|        P2| Cabang Bandung|  Espresso|  1|       15000|2026-11-05 16:55:39|
|        P3| Cabang Jakarta|     Latte|  1|       25000|2026-11-05 16:56:39|
|        P4|Cabang Surabaya|Kopi Hitam|  3|       10000|2026-11-05 16:56:39|
+----------+---------------+----------+---+------------+-------------------+

